# 1.7 候選分數怎麼變成機率？


沿用「貓、看、狗」三個候選，假設分數為 `[1,2,-1]`。我們想保留「看最高、狗最低」的偏好，又希望得到能拿來抽字的比例。原分數有負數、總和不為 1，不能直接當機率。

先用指數函式 `exp` 把每個分數換成正數，再除以正數的總和，這一整套叫 **softmax**。`exp(z)` 是 e 的 z 次方，e 約 2.718；三個正數約為 `[2.718,7.389,0.368]`，總和 10.475。逐個相除，得到約 `[0.2595,0.7054,0.0351]`，看仍最受偏好。

把全部分數加同一常數，指數只會共同多一個乘數，比例化時會抵消。因此可以先減最高分 2，使分數成 `[-1,0,-3]`，避免指數過大，再得到相同機率。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

z = torch.tensor([1.0, 2.0, -1.0])
weights = (z - z.max()).exp()
p = weights / weights.sum()
print(weights)
print(p, p.sum().item())
print((z + 100).softmax(dim=0))
assert torch.allclose(p, z.softmax(dim=0))

第一行印約 `[0.368,1,0.050]`，第二行是三個機率與總和 1，第三行將全部分數加 100 後，仍得到同一分配。最後將手算與內建 `softmax` 核對。

`dim=0` 指本例唯一的候選軸。若有多筆分數 `[筆數,候選數]`，應沿最後的候選軸換算，讓每題各自加總為 1。不同題目不能共用一份機率總額。

softmax 只整理偏好，沒有讀正確答案。若狗的錯誤分數最高，它仍會給狗最多機會。下一節才用答案判定這份機率好不好。

練習把貓的分數 1 改成 3，其他不動。三機率變成約 `[0.7214,0.2654,0.0132]`：看的分數沒改，機率卻降低，因為大家共用同一個分母。

<details>
<summary>補充與重做</summary>

機率和對數背景見 [W.4](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4)。`allclose` 容許很小的浮點誤差；`assert` 通過時不印額外訊息。此段沒有改任何參數，只把同一組分數換成比例。

</details>
